In [ ]:
import cv2
import time
import numpy as np
from collections import deque
from tensorflow.keras.models import load_model

IMG_SIZE = 100

emotion_labels = [
    "surprise",
    "fear",
    "disgust",
    "happy",
    "sad",
    "angry",
    "neutral"
]

# Open camera first
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Cannot open camera.")
    exit()

# preheat camera
time.sleep(1)

# test if it could read
ret, frame = cap.read()
print("Initial camera read:", ret)

if not ret:
    print("Camera opened but cannot read frame.")
    cap.release()
    exit()

# reload model
emotion_model = load_model(
    "best_rafdb_resnet_model.keras",
    compile=False
)

print("Model loaded successfully.")
print("Model input shape:", emotion_model.input_shape)

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

pred_queue = deque(maxlen=5)

while True:
    ret, frame = cap.read()

    if not ret or frame is None:
        print("Cannot read frame during loop.")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=5,
        minSize=(60, 60)
    )

    for (x, y, w, h) in faces:
        face = frame[y:y+h, x:x+w]

        if face.size == 0:
            continue

        # BGR -> RGB
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))

        
        face = face.astype("float32")
        face = np.expand_dims(face, axis=0)

        pred = emotion_model.predict(face, verbose=0)[0]

        pred_queue.append(pred)
        avg_pred = np.mean(pred_queue, axis=0)

        emotion_id = np.argmax(avg_pred)
        confidence_score = np.max(avg_pred)

        if confidence_score < 0.35:
            label = "unknown"
        else:
            label = f"{emotion_labels[emotion_id]} {confidence_score:.2f}"

        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

        cv2.putText(
            frame,
            label,
            (x, max(30, y - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

    cv2.imshow("RAF-DB ResNet-style CNN Emotion Recognition", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

Initial camera read: True
Model loaded successfully.
Model input shape: (None, 100, 100, 3)
